In [ ]:
%spark.pyspark
from pyspark.sql import functions as F

users = spark.createDataFrame(
    [
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"),
    ],
    ["user_id", "city"]
)

orders = spark.createDataFrame(
    [
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0),
    ],
    ["order_id", "user_id", "product_id", "qty", "price"]
)

products = spark.createDataFrame(
    [
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"),
    ],
    ["product_id", "product_name"]
)

users.show()
orders.show()
products.show()

In [ ]:
%spark.pyspark
orders = orders.withColumn("revenue", F.col("qty") * F.col("price"))

orders.show()

In [ ]:
%spark.pyspark
df = orders.join(users, on="user_id", how="inner") \
           .join(products, on="product_id", how="inner")

df.show()

In [ ]:
%spark.pyspark
mart = df.groupBy("city", "product_id", "product_name") \
         .agg(
             F.count("order_id").alias("orders_cnt"),
             F.sum("qty").alias("qty_sum"),
             F.sum("revenue").alias("revenue_sum")
         )

mart.show()

In [ ]:
%spark.pyspark
from pyspark.sql.window import Window

window = Window.partitionBy("city").orderBy(F.desc("revenue_sum"))

mart_top = mart.withColumn("rank", F.row_number().over(window)) \
               .filter(F.col("rank") <= 2) \
               .drop("rank")

mart_top.show()

In [ ]:
%spark.pyspark
path = "/tmp/sandbox_zeppelin/mart_city_top_products/"

mart_top.write.mode("overwrite").parquet(path)

In [ ]:
%spark.pyspark
df_check = spark.read.parquet("/tmp/sandbox_zeppelin/mart_city_top_products/")

df_check.show()

In [ ]:
%spark.pyspark
